# Main Colab: Bangla hate speech detection with transformer ensembles

Runs the whole thesis pipeline from the GitHub repository. Use **Runtime > Change runtime type > T4 GPU** for Section 5.

* Every job is **resumable** (finished runs are skipped) and **independent**, so you can run different jobs in different Colab accounts and merge the predictions at the end (Section 6).
* Sections: 1 setup, 2 data, 3 helpers, 4 classical baselines, 5 transformers (Study 1/2/3), 6 merge, 7 analysis.
* **Only want to check the thesis numbers?** Run Section 1 and then **Section 0** (no GPU, no data, about 1 minute).
* The own dataset is private: upload it yourself (Section 2). Without it you can still run Study 1 (Option S) and Section 0.

## 1. Setup (always run first)

In [ ]:
REPO_URL = "https://github.com/abdullah-mohammed0000/bangla-hate-speech-ensembles.git"
import os, sys, subprocess
if not os.path.exists("/content/repo"):
    subprocess.run(["git", "clone", "-q", REPO_URL, "/content/repo"], check=True)
else:
    subprocess.run(["git", "-C", "/content/repo", "pull", "-q"], check=True)
%cd /content/repo
!pip -q install -r requirements.txt -r requirements-train.txt
sys.path.insert(0, "/content/repo/src")
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - change the runtime to T4 GPU for Section 5")

## 0. Check the thesis numbers from the released predictions (optional, CPU, ~1 minute)
Re-creates every result table and the significance tests from `results/predictions.zip`, and checks 13 headline numbers against the thesis (`ALL REPRODUCED`).

In [ ]:
!unzip -qo results/predictions.zip -d results/
!python scripts/verify_release.py --predictions results/predictions
!python scripts/analyze.py --predictions results/predictions --out results/tables_check

## 2. Data
Choose **one** option and set `DATA_MODE`.

* **Option A (recommended):** upload the four raw zips (`archive__3_.zip` = BD-SHS, `Banhate.zip`, `BanglaHateBert-main.zip`, `Own Data.zip`). The script rebuilds the splits and prints **OK** when they are identical to the thesis files.
* **Option B:** upload `merged_splits.zip` (the nine split CSVs, e.g. made earlier by Option A in another account); the files are hash-checked when training starts.
* **Option S (Study 1 only):** upload only the BD-SHS zip (`archive__3_.zip`) and use the Study 1 cell at the end of Section 5.

In [ ]:
DATA_MODE = "A"        # "A" = raw zips, "B" = merged_splits.zip, "S" = BD-SHS zip only (Study 1)
from google.colab import files
import shutil, zipfile, glob
os.makedirs("data/raw", exist_ok=True); os.makedirs("data/splits", exist_ok=True)
up = files.upload()
for name, content in up.items():
    dst = ("data/" if DATA_MODE == "B" else "data/raw/") + name
    open(dst, "wb").write(content)
if DATA_MODE == "A":
    !python scripts/prepare_data.py --raw data/raw --out data/splits
elif DATA_MODE == "B":
    for zname in up:
        with zipfile.ZipFile("data/" + zname) as z: z.extractall("data/splits_tmp")
    for f in glob.glob("data/splits_tmp/**/*.csv", recursive=True): shutil.copy(f, "data/splits/")
    shutil.rmtree("data/splits_tmp", ignore_errors=True)
    print(sorted(os.listdir("data/splits")))
    from bhs.data import load_splits
    load_splits("data/splits")   # stops with an error if a file differs from the thesis files
    print("OK: the split files have the thesis hashes.")
else:
    print("Study 1 only: BD-SHS uploaded to data/raw - go to the last cell of Section 5.")

## 3. Helpers: save progress / merge results

In [ ]:
import os, sys, io, glob, shutil, zipfile, time
import pandas as pd
from google.colab import files
sys.path.insert(0, "/content/repo/src")

def save_progress(tag="progress"):
    """Zip all predictions + logs of this session and download it (keep these zips: they are the results)."""
    zp = f"/content/predictions_{tag}_{time.strftime('%Y%m%d_%H%M')}.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        for f in glob.glob("runs/predictions/*"):
            if os.path.isfile(f): z.write(f, os.path.basename(f))
    print("saved", zp); files.download(zp)

def merge_zips():
    """Upload prediction zips from other Colab accounts; their files are copied into runs/predictions.
    The run logs results*.csv have the same name in every account, so their rows are appended instead."""
    os.makedirs("runs/predictions", exist_ok=True)
    for name, content in files.upload().items():
        with zipfile.ZipFile(io.BytesIO(content)) as z: z.extractall("runs/_merge_tmp")
    n = 0
    for f in glob.glob("runs/_merge_tmp/**/*", recursive=True):
        if not os.path.isfile(f): continue
        base = os.path.basename(f); dst = "runs/predictions/" + base
        if base.startswith("results") and base.endswith(".csv") and os.path.exists(dst):
            pd.concat([pd.read_csv(dst), pd.read_csv(f)]).drop_duplicates().to_csv(dst, index=False)
        elif not os.path.exists(dst):
            shutil.copy(f, dst); n += 1
    shutil.rmtree("runs/_merge_tmp", ignore_errors=True); print("copied", n, "new files")

## 4. Classical baselines (SVM, BiLSTM) - Studies 1 and 2 (needs Option A or B)

In [ ]:
!python scripts/train_classical.py --regime A B
save_progress("classical")

## 5. Transformers (T4 GPU)
Seeds 42, 43, 44 for every job. Suggested split across accounts (each job group is independent):

| Group | Models | Regimes | Study |
|---|---|---|---|
| 5a | banglabert_norm, xlmr | A B | 1, 2 |
| 5b | banglabert, indicbertv2, mbert | A B | 1, 2 |
| 5c | banglabert_large_norm | A B | 1, 2 (the large model needs the most GPU time - one seed per account is fine: change `--seed`) |
| 5d | mbert | C | 2 (size-matched control) |
| 5e | banglabert_norm, xlmr, banglabert_large_norm | O | 3 (own dataset only) |

Run **only the group(s) you need in this session**; each cell downloads its predictions at the end.

In [ ]:
# 5a - BanglaBERT + normalizer and XLM-R, Single and Combine
!python scripts/train_transformer.py --model banglabert_norm xlmr --regime A B --seed 42 43 44
save_progress("5a")

In [ ]:
# 5b - BanglaBERT, IndicBERTv2, mBERT
!python scripts/train_transformer.py --model banglabert indicbertv2 mbert --regime A B --seed 42 43 44
save_progress("5b")

In [ ]:
# 5c - BanglaBERT-large + normalizer (change --seed to split over sessions, e.g. --seed 42)
!python scripts/train_transformer.py --model banglabert_large_norm --regime A B --seed 42 43 44
save_progress("5c")

In [ ]:
# 5d - size-matched control (mBERT, BD-SHS resampled to the size of the merged set)
!python scripts/train_transformer.py --model mbert --regime C --seed 42 43 44
save_progress("5d")

In [ ]:
# 5e - Study 3: own dataset only
!python scripts/train_transformer.py --model banglabert_norm xlmr banglabert_large_norm --regime O --seed 42 43 44
save_progress("5e")

In [ ]:
# Study 1 only (Option S): just the public BD-SHS files, no other corpus needed.
# Add the other E6 members (xlmr banglabert_large_norm) to the --model list to get E6.
!python scripts/train_transformer.py --model banglabert_norm --seed 42 43 44 --bdshs-only data/raw
save_progress("study1")

## 6. Merge predictions from other Colab accounts (optional)

In [ ]:
merge_zips()    # upload the predictions_*.zip files downloaded in the other accounts
print(len(glob.glob("runs/predictions/probs_*.npy")), "prediction files")

## 7. Analysis of your own runs (CPU, ~1 minute)
The training scripts store the test labels next to the predictions, so this works after any set of jobs; parts whose runs are missing are skipped.

In [ ]:
!python scripts/analyze.py --predictions runs/predictions --out results/tables_rerun
# compare with the thesis numbers (differences of a few thousandths are expected after re-training; see docs/REPRODUCING.md)
!python scripts/verify_release.py --predictions runs/predictions

In [ ]:
# download the tables
!zip -qr tables_rerun.zip results/tables_rerun
files.download("tables_rerun.zip")